# KWISMO — Notebook 01 : Analyse Exploratoire & Visualisations Avancées (EDA Global)

Ce notebook effectue l'analyse statistique et la visualisation graphique globale des jeux de données **Modèle A** (Scoring comportemental & temporel) et **Modèle B** (NLP, argots & détection d'arnaques).

### Principes de Robustesse :
- Parcourt l'ensemble des répertoires du projet sans blocage en cas d'absence d'un fichier intermédiaire.
- Génère des graphiques clairs et synthétiques pour le **Modèle A** (distributions, corrélations, vélocités) et le **Modèle B** (volumétrie texte, n-grams, dialectes).

In [ ]:
# 1. Initialisation de l'environnement et détection non-bloquante des jeux de données
import os
import sys
import json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.size"] = 10

# Détection racine projet
current_dir = Path.cwd()
repo_root = current_dir
for candidate in [current_dir, current_dir.parent, current_dir.parent.parent]:
    if (candidate / "data").exists() or (candidate / "src").exists():
        repo_root = candidate.resolve()
        break
    elif (candidate / "kwismo-ai" / "src").exists():
        repo_root = (candidate / "kwismo-ai").resolve()
        break

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(f"Dossier racine identifié : {repo_root}")

## PARTIE 1 : Analyse Exploratoire du Modèle A (Scoring Comportemental & Temporel)

In [ ]:
model_a_path = repo_root / "data" / "processed" / "model_a_dataset.csv"

if not model_a_path.exists():
    print("⚙️ Dataset Modèle A introuvable. Génération automatique...")
    try:
        from src.data.generate_model_a_data import generate_dataset
        df_a = generate_dataset(num_samples=1200)
    except Exception as err:
        print(f"⚠️ Impossible de générer le dataset Modèle A : {err}")
        df_a = None
else:
    try:
        df_a = pd.read_csv(model_a_path)
    except Exception as err:
        print(f"⚠️ Erreur lors de la lecture du fichier Modèle A : {err}")
        df_a = None

if df_a is not None:
    print(f"✅ Dataset Modèle A chargé ({len(df_a)} numéros)")
    display(df_a.head(3))
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # 1. Distribution de la variable cible (Label)
    sns.countplot(data=df_a, x="label", ax=axes[0, 0], palette=["#2A9D8F", "#E63946"])
    axes[0, 0].set_title("1. Distribution de la Classe Cible (0 = Sécurisé, 1 = Frauduleux)", fontsize=11, fontweight="bold")
    axes[0, 0].set_xlabel("Label de Risque")
    axes[0, 0].set_ylabel("Nombre de Numéros")
    
    # 2. Relation Vérifications vs Signalements
    sns.scatterplot(data=df_a, x="nombre_verifications", y="nombre_signalements", hue="label", style="label", palette=["#2A9D8F", "#E63946"], ax=axes[0, 1])
    axes[0, 1].set_title("2. Vérifications vs Signalements enregistrés", fontsize=11, fontweight="bold")
    axes[0, 1].set_xlabel("Nombre de vérifications d'utilisateurs")
    axes[0, 1].set_ylabel("Nombre de signalements rédigés")
    
    # 3. Ratio de Paresse (Ratio Verif / Signalement)
    sns.kdeplot(data=df_a, x="ratio_verif_signalement", hue="label", fill=True, common_norm=False, palette=["#2A9D8F", "#E63946"], ax=axes[1, 0])
    axes[1, 0].set_title("3. Densité du Ratio de Paresse (Vérifications / Signalements)", fontsize=11, fontweight="bold")
    axes[1, 0].set_xlabel("Ratio Verif / Signalement")
    
    # 4. Matrice de Corrélation des Variables Comportementales
    num_cols = df_a.select_dtypes(include=[np.number]).columns
    corr = df_a[num_cols].corr()
    sns.heatmap(corr, annot=False, cmap="coolwarm", ax=axes[1, 1], cbar=True)
    axes[1, 1].set_title("4. Matrice de Corrélation (Heatmap des Caractéristiques)", fontsize=11, fontweight="bold")
    
    plt.tight_layout()
    plt.show()
else:
    print("⏩ Modèle A ignoré (fichier indisponible).")

## PARTIE 2 : Analyse Exploratoire du Modèle B (NLP, Dialectes & Messages)

In [ ]:
possible_b_paths = [
    repo_root / "data" / "processed" / "model_b_augmented.jsonl",
    repo_root / "data" / "processed" / "model_b_clean.jsonl",
    repo_root / "data" / "processed" / "legit_examples.jsonl",
    repo_root / "data" / "raw" / "scraped" / "messages.jsonl"
]

records_b = []
for p in possible_b_paths:
    if p.exists():
        try:
            with open(p, "r", encoding="utf-8") as f:
                for line in f:
                    if line.strip():
                        item = json.loads(line)
                        item["_source_file"] = p.name
                        records_b.append(item)
        except Exception:
            continue

if records_b:
    df_b = pd.DataFrame(records_b)
    print(f"✅ Dataset Modèle B chargé ({len(df_b)} messages issus de {len(possible_b_paths)} sources)")
    display(df_b.head(3))
    
    text_col = None
    for col in ["description", "texte", "text", "content"]:
        if col in df_b.columns:
            text_col = col
            break
            
    if text_col:
        df_b["longueur_caracteres"] = df_b[text_col].fillna("").apply(len)
        df_b["nombre_mots"] = df_b[text_col].fillna("").apply(lambda t: len(t.split()))
        
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
        
        sns.histplot(df_b["nombre_mots"], kde=True, color="#2E86AB", ax=ax1, bins=25)
        ax1.set_title("Distribution du Nombre de Mots par Message", fontsize=11, fontweight="bold")
        ax1.set_xlabel("Nombre de mots")
        
        sns.countplot(data=df_b, x="_source_file", ax=ax2, palette="mako")
        ax2.set_title("Volume de Messages par Fichier Source", fontsize=11, fontweight="bold")
        ax2.set_xlabel("Fichier Source")
        ax2.set_ylabel("Nombre de Messages")
        plt.xticks(rotation=15)
        
        plt.tight_layout()
        plt.show()
else:
    print("⏩ Modèle B ignoré (fichiers NLP indisponibles).")